# Nighttime Lights in Myanmar

This notebook has been prepared for the Myanmar Economic Monitor (Spring 2026). It uses data from the NASA VIIRS BlackMarble dataset available from 2012 till March 2026.

**Sections:**

1. **National Nighttime Lights Trends**
   - 1.1 Annual Trends in Nightlights
   - 1.2 Quarterly Trends in Nightlights
   - 1.3 Annual % Change in Nightlights
   - 1.4 Monthly Trends in Nightlights (comparing 2024, 2025, 2026)

2. **Nightlights in Industrial Areas**
   - 2.1 Quarterly Trends in Industrial vs Non-Industrial Areas
   - 2.2 Percentage Change in Industrial vs Non-Industrial Areas
   - 2.3 Monthly Trends in Industrial Areas (comparing 2024, 2025, 2026)

3. **Nightlights in Regions**
   - 3.1 Quarterly Trends in Nightlights in Regions
   - 3.2 Monthly Trends in Nightlights in Regions (comparing 2024, 2025, 2026)
   - 3.3 Monthly Trends in Regional Industrial Areas (comparing 2024, 2025, 2026)

4. **Spatial Distribution of Nightlights**
   - 4.1 Quarterly Nightlights in Regions
   - 4.2 Quarterly Nightlights in Districts

In [135]:
from pathlib import Path

import altair as alt
import attaviz
import geopandas as gpd
import numpy as np
import pandas as pd

In [136]:
# Import visualization functions
%load_ext autoreload
%autoreload 2  
# sys.path.append('../../../src/')
from utils import *
from visuals import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [137]:
DATA_FOLDER = Path("../../../data/")
NTL_FOLDER = DATA_FOLDER / "ntl" / "collection2"

from calendar import month_abbr, month_name
from datetime import datetime

current_year = datetime.now().year
prev_year = current_year - 1
max_month = datetime.now().month - 2
report_period_label = f"Jan-{month_abbr[max_month]}"
report_period_subtitle = f"January-{month_name[max_month]}"

In [138]:
# --- Load NTL data ---
ntl_annual_adm0 = pd.read_csv(NTL_FOLDER / "annual/ntl_mmr_adm0_annual.csv")
ntl_annual_adm1 = pd.read_csv(NTL_FOLDER / "annual/ntl_mmr_adm1_annual.csv")
ntl_monthly_adm0 = pd.read_csv(NTL_FOLDER / "monthly/ntl_mmr_adm0_monthly.csv")
ntl_monthly_adm1 = pd.read_csv(NTL_FOLDER / "monthly/ntl_mmr_adm1_monthly.csv")
ntl_monthly_adm2 = pd.read_csv(NTL_FOLDER / "monthly/ntl_mmr_adm2_monthly.csv")
ntl_annual_ind_5km = pd.read_csv(NTL_FOLDER / "annual/ntl_mmr_admsez_annual.csv")

# --- Load boundaries ---
mmr_adm0 = gpd.read_file(DATA_FOLDER / "Boundaries" / "mmr_polbnda_adm0_250k_mimu.shp")
mmr_adm1 = gpd.read_file(DATA_FOLDER / "Boundaries" / "mmr_polbnda2_adm1_250k_mimu.shp")
mmr_adm2 = gpd.read_file(
    DATA_FOLDER / "boundaries" / "mmr_polbnda_adm2_250k_mimu_1.shp"
)
mmr_adm3 = gpd.read_file(
    DATA_FOLDER / "boundaries_old" / "mmr_polbnda_adm3_250k_mimu_20240215.shp"
)

# --- Load industrial zones ---
industrial_zones = gpd.read_file(
    DATA_FOLDER / "boundaries_old" / "industrial__special_economic_zones_sept2019.shp"
)
industrial_zones = industrial_zones.sjoin(
    mmr_adm3[
        [
            "ADM1_PCODE",
            "ADM1_EN",
            "ADM2_EN",
            "ADM2_PCODE",
            "ADM3_EN",
            "ADM3_PCODE",
            "geometry",
        ]
    ]
)

In [139]:
def normalize_adm1_columns(df):
    df = df.copy()
    if "ST" in df.columns:
        if "ADM1_EN" in df.columns:
            df["ADM1_EN"] = df["ADM1_EN"].fillna(df["ST"])
            df = df.drop(columns=["ST"])
        else:
            df = df.rename(columns={"ST": "ADM1_EN"})
    for old_col in ["st_code", "ST_PCODE"]:
        if old_col in df.columns:
            if "ADM1_PCODE" in df.columns:
                df["ADM1_PCODE"] = df["ADM1_PCODE"].fillna(df[old_col])
                df = df.drop(columns=[old_col])
            else:
                df = df.rename(columns={old_col: "ADM1_PCODE"})
    return df


ntl_monthly_adm1 = normalize_adm1_columns(ntl_monthly_adm1)
ntl_monthly_adm0 = normalize_adm1_columns(ntl_monthly_adm0)

In [140]:
mmr_adm1.rename(
    columns={"ST": "ADM1_EN", "ST_PCODE": "ADM1_PCODE", "st_code": "ADM1_PCODE"},
    inplace=True,
)

In [141]:
mmr_adm2.rename(
    columns={
        "ST": "ADM1_EN",
        "ST_PCODE": "ADM1_PCODE",
        "st_code": "ADM1_PCODE",
        "DT": "ADM2_EN",
        "DT_PCODE": "ADM2_PCODE",
    },
    inplace=True,
)

In [142]:
# --- Prepare all derived datasets ---
ntl_monthly_adm0["date"] = pd.to_datetime(ntl_monthly_adm0["date"])
ntl_monthly_adm0["year_col"] = ntl_monthly_adm0["date"].dt.year
ntl_monthly_adm0["month"] = ntl_monthly_adm0["date"].dt.month

ntl_monthly_adm1["date"] = pd.to_datetime(ntl_monthly_adm1["date"])
ntl_monthly_adm1["year_col"] = ntl_monthly_adm1["date"].dt.year
ntl_monthly_adm1["month"] = ntl_monthly_adm1["date"].dt.month

# Partial-year corrected totals (Jan through max_month)
ntl_corrected = prepare_ntl_corrected(ntl_monthly_adm0, max_month)

# Annual YoY with partial-year overlay
annual_yoy = prepare_annual_yoy_with_partial(ntl_annual_adm0, ntl_corrected, "ntl_sum")
annual_ind_yoy = prepare_annual_yoy_with_partial(
    ntl_annual_adm0, ntl_corrected, "ntl_ind_5km_sum"
)
annual_noind_yoy = prepare_annual_yoy_with_partial(
    ntl_annual_adm0, ntl_corrected, "ntl_noind_5km_sum"
)

# Monthly comparison (3 years)
ntl_comparison = prepare_monthly_comparison(
    ntl_monthly_adm0, [prev_year - 1, prev_year, current_year]
)

# Industrial vs non-industrial melted
ntl_ind_melted = prepare_industrial_comparison_melted(
    ntl_monthly_adm0, [prev_year - 1, prev_year, current_year]
)

# Regional industrial data
df_regional_ind, region_order_list = prepare_regional_industrial_data(ntl_monthly_adm1)
ntl_regional_comparison = prepare_regional_comparison(
    ntl_monthly_adm1, [prev_year - 1, prev_year, current_year], region_order_list
)

# Earthquake period data
ntl_earthquake_period = prepare_earthquake_period_data(ntl_monthly_adm1, mmr_adm1)

# Subnational monthly pct changes
monthly_pct_changes = subnational_monthly_pct_change(
    df=ntl_monthly_adm2,
    region_col="ADM2_EN",
    baseline_year=prev_year,
    current_year=current_year,
    date_col="date",
)
monthly_pct_changes = mmr_adm2[["ADM2_EN", "geometry"]].merge(
    monthly_pct_changes, on="ADM2_EN", how="inner"
)

## 1. National Nighttime Lights Trends

This section highlights the natiobnal change in nightlights. The last available anual dataset is for 2025. 

### 1.1 Annual Trends in Nightlights

In [143]:
# Fig 1.1: Annual Trends in Nightlights
chart = (
    alt.Chart(ntl_annual_adm0.assign(year=lambda d: pd.to_datetime(d["date"]).dt.year))
    .mark_line(point=True, color="#025288", strokeWidth=2)
    .encode(
        x=alt.X(
            "year:O", title="Year", axis=alt.Axis(labelFontSize=9, titleFontSize=10)
        ),
        y=alt.Y(
            "ntl_sum:Q",
            title="",
            axis=alt.Axis(
                labelFontSize=9,
                titleFontSize=10,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
        ],
    )
    .properties(
        width=500,
        height=260,
        title=alt.TitleParams(
            text="Annual National Nighttime Lights", fontSize=14, anchor="start"
        ),
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)

Myanmar saw an earthquake in 2025 which likely contributed to the reduction in overall nightlights. 

### 1.2 Quarterly Trends in Nightlights

In [144]:
# Fig 1.2: Quarterly Trends in Nightlights
chart = (
    alt.Chart(ntl_corrected.assign(year=lambda d: d["year_col"].astype(str)))
    .mark_bar(color="#025288")
    .encode(
        x=alt.X(
            "year:N", title="Year", axis=alt.Axis(labelFontSize=9, titleFontSize=10)
        ),
        y=alt.Y(
            "ntl_sum:Q",
            title="Luminosity",
            axis=alt.Axis(
                labelFontSize=9,
                titleFontSize=10,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        tooltip=[
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
        ],
    )
    .properties(
        width=500,
        height=260,
        title=alt.TitleParams(
            text=f"{report_period_label} Nightlight Trend", fontSize=14, anchor="start"
        ),
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)

However, if the average lights in the period of January to March are compared, it is seen that 2026 shows signs of recovery with the second highest nightlights since 2012, closely following 2023. 

### 1.3 Annual % Change in Nightlights

In [145]:
# Fig 1.3: Annual % Change in Nightlights
REPORT_MAX_MONTH = max_month
ntl_corrected_report = prepare_ntl_corrected(ntl_monthly_adm0, REPORT_MAX_MONTH)
annual_report = prepare_annual_yoy_with_partial(
    ntl_annual_adm0, ntl_corrected_report, "ntl_sum"
)

full = (
    annual_report[["year", "pct_change_full_year"]]
    .dropna()
    .query("year < @current_year")
    .rename(columns={"pct_change_full_year": "pct_change"})
    .assign(series="% change from PY", series_order=0)
)
partial = (
    annual_report.loc[
        annual_report["year"].isin([prev_year, current_year]),
        ["year", "pct_change_9months"],
    ]
    .dropna()
    .rename(columns={"pct_change_9months": "pct_change"})
    .assign(
        series=lambda d: np.where(
            d["year"].eq(prev_year),
            f"{report_period_label} {prev_year}",
            f"{report_period_label} {current_year}",
        ),
        series_order=lambda d: np.where(d["year"].eq(prev_year), 1, 2),
    )
)
annual_ntl_chart_data = (
    pd.concat([full, partial], ignore_index=True)
    .assign(
        year=lambda d: d["year"].astype(int),
        label=lambda d: d["pct_change"].map(lambda v: f"{v:.1f}%"),
    )
    .sort_values(["year", "series_order"])
)

PROJECT_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists() and (p / "notebooks").exists()
)
NTL_REPORT_DIR = PROJECT_ROOT / "docs" / "ntl_report"
NTL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
annual_ntl_chart_data.to_csv(NTL_REPORT_DIR / "fig-1-3-annual-ntl-yoy.csv", index=False)
bar_size = 24
y_axis_min = min(-15, annual_ntl_chart_data["pct_change"].min() - 5)

series_domain = [
    "% change from PY",
    f"{report_period_label} {prev_year}",
    f"{report_period_label} {current_year}",
]
base = alt.Chart(annual_ntl_chart_data).encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelFontSize=9, titleFontSize=10)),
    y=alt.Y(
        "pct_change:Q",
        title="Luminosity (ntl_sum) % Change",
        scale=alt.Scale(domainMin=y_axis_min),
        axis=alt.Axis(labelFontSize=9, titleFontSize=10, grid=True, gridDash=[2, 2]),
    ),
    tooltip=[
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("series:N", title="Measure"),
        alt.Tooltip("pct_change:Q", format="+.1f", title="% Change"),
    ],
)
full_bars = (
    base.transform_filter(alt.datum.series == "% change from PY")
    .mark_bar(size=bar_size)
    .encode(
        color=alt.condition(
            alt.datum.pct_change < 0, alt.value("#24768E"), alt.value("#754493")
        )
    )
)
partial_bars = (
    base.transform_filter(alt.datum.series != "% change from PY")
    .mark_bar(size=bar_size)
    .encode(
        color=alt.Color(
            "series:N",
            title="",
            scale=alt.Scale(domain=series_domain[1:], range=["#24768E", "#754493"]),
            legend=alt.Legend(labelFontSize=9, symbolSize=80),
        ),
        opacity=alt.value(0.58),
    )
)
labels = base.mark_text(
    fontSize=8, color="#666666", dy=alt.expr("datum.pct_change >= 0 ? -5 : 12")
).encode(text="label:N")
zero = (
    alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#8A8A8A").encode(y="zero:Q")
)
chart = (full_bars + partial_bars + labels + zero).properties(
    width=434,
    height=280,
    title=alt.TitleParams(
        text="Myanmar Nighttime Lights - Annual % Change",
        subtitle=f"Full-year YoY through {prev_year}; {report_period_label} change for {prev_year} and {current_year}",
        fontSize=14,
        subtitleFontSize=10,
        anchor="start",
    ),
)
chart = attaviz.add_caption(chart, "Source: NASA BlackMarble")
chart.save(NTL_REPORT_DIR / "fig-1-3-annual-ntl-yoy.png", scale_factor=3)
chart

alt.VConcatChart(...)

2025 saw a 7.8% reduction in light compared to 2024. However, in the first four months of 2025 (earthquake period), there was a ~29% reduction in light. In comparison, there is a 47% increase in light in 2026 compared to 2025. 

### 1.4 Monthly Trends in Nightlights (comparing 2024, 2025, 2026)

In [146]:
# Fig 1.4b: Monthly Trends - Comparative Lines (2024 vs 2025 vs 2026)
NTL_REPORT_DIR = Path("../../../docs/ntl_report")
NTL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
NTL_REPORT_XLSX = NTL_REPORT_DIR / "ntl-report-data.xlsx"


def write_ntl_sheet(df, sheet_name):
    mode = "a" if NTL_REPORT_XLSX.exists() else "w"
    kwargs = {"engine": "openpyxl", "mode": mode}
    if mode == "a":
        kwargs["if_sheet_exists"] = "replace"
    with pd.ExcelWriter(NTL_REPORT_XLSX, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)


month_labels = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
ntl_monthly_lines = (
    ntl_comparison[["date", "month", "year_col", "ntl_sum"]]
    .assign(
        date=lambda d: pd.to_datetime(d["date"]),
        year_col=lambda d: d["year_col"].astype(str),
        month_name=lambda d: d["month"].map(lambda m: month_labels[int(m) - 1]),
    )
    .sort_values(["year_col", "month"])
)
write_ntl_sheet(ntl_monthly_lines, "fig_1_4b_monthly_ntl")

chart = (
    alt.Chart(ntl_monthly_lines)
    .mark_line(point=False, strokeWidth=2)
    .encode(
        x=alt.X(
            "month:O",
            title="Month",
            sort=list(range(1, 13)),
            axis=alt.Axis(
                labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value - 1]",
                labelFontSize=9,
                titleFontSize=10,
            ),
        ),
        y=alt.Y(
            "ntl_sum:Q",
            title="NTL Sum",
            axis=alt.Axis(
                labelFontSize=9,
                titleFontSize=10,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        color=alt.Color(
            "year_col:N",
            title="",
            scale=alt.Scale(range=["#025288", "#E3A763", "#754493"]),
            legend=alt.Legend(labelFontSize=9, symbolSize=70),
        ),
        tooltip=[
            alt.Tooltip("year_col:N", title="Year"),
            alt.Tooltip("month_name:N", title="Month"),
            alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
        ],
    )
    .properties(
        width=500,
        height=240,
        title=alt.TitleParams(
            text="Monthly Nighttime Lights",
            subtitle="Monthly NTL sum, 2024-2026",
            fontSize=14,
            subtitleFontSize=10,
            anchor="start",
            offset=6,
        ),
    )
)
chart = attaviz.add_caption(chart, "Source: NASA BlackMarble")
chart.save(NTL_REPORT_DIR / "fig-1-4b-monthly-nighttime-lights.png", scale_factor=3)
chart

alt.VConcatChart(...)

The monthly nightlights in 2026 surpassed 2024 for January and February. There was a slight dip in lights in March 2026 compared to 2024. 

## 2. Nightlights in Industrial Areas

There are 97 industrial zones in Myanamr classified as Special Economic Zones. The majority are in Yangon. The analysis for nightlights in industrial zones is done by assuming a 5km buffer zone around the points. 

In [147]:
sez = gpd.read_file(
    "../../../data/boundaries_old/industrial__special_economic_zones_sept2019.shp"
)
sez.explore()

### 2.1 Quarterly Trends in Industrial vs Non-Industrial Areas

In [148]:
# Fig 2.1: Quarterly Trends in Industrial vs Non-Industrial Areas
ind_quarter_data = (
    ntl_corrected[["year_col", "ntl_ind_5km_sum", "ntl_noind_5km_sum"]]
    .melt(
        id_vars="year_col",
        value_vars=["ntl_ind_5km_sum", "ntl_noind_5km_sum"],
        var_name="type",
        value_name="ntl_sum",
    )
    .assign(
        type=lambda d: d["type"].replace(
            {
                "ntl_ind_5km_sum": "Industrial Zones",
                "ntl_noind_5km_sum": "Non-Industrial Zones",
            }
        ),
        year=lambda d: d["year_col"].astype(str),
    )
)
chart = (
    alt.Chart(ind_quarter_data)
    .mark_bar(color="#025288")
    .encode(
        x=alt.X(
            "year:N", title="Year", axis=alt.Axis(labelFontSize=9, titleFontSize=10)
        ),
        y=alt.Y(
            "ntl_sum:Q",
            title="Luminosity",
            axis=alt.Axis(
                labelFontSize=9,
                titleFontSize=10,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        tooltip=[
            alt.Tooltip("type:N", title="Type"),
            alt.Tooltip("year:N", title="Year"),
            alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
        ],
    )
    .properties(width=250, height=240)
    .facet(column=alt.Column("type:N", title=""))
    .properties(
        title=alt.TitleParams(
            text=f"Industrial and Non-Industrial {report_period_label} Nighttime Lights",
            fontSize=14,
            anchor="start",
        )
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)

The nighttime lights in the first four months of 2026 has surpassed that of 2024 for both Industrial and non industrial zones. 

### 2.2 Percentage Change in Industrial vs Non-Industrial Areas

In [149]:
# Fig 2.2: Percentage Change in Industrial vs Non-Industrial Areas
REPORT_MAX_MONTH = max_month
ntl_corrected_report = prepare_ntl_corrected(ntl_monthly_adm0, REPORT_MAX_MONTH)
annual_ind_report = prepare_annual_yoy_with_partial(
    ntl_annual_adm0, ntl_corrected_report, "ntl_ind_5km_sum"
)
annual_noind_report = prepare_annual_yoy_with_partial(
    ntl_annual_adm0, ntl_corrected_report, "ntl_noind_5km_sum"
)


def annual_pct_chart_data(df, zone):
    full = (
        df[["year", "pct_change_full_year"]]
        .dropna()
        .query("year < @current_year")
        .rename(columns={"pct_change_full_year": "pct_change"})
        .assign(zone=zone, series="% change from PY")
    )
    partial = (
        df.loc[df["year"].eq(current_year), ["year", "pct_change_9months"]]
        .dropna()
        .rename(columns={"pct_change_9months": "pct_change"})
        .assign(zone=zone, series=f"{REPORT_MAX_MONTH}M % change from PY")
    )
    return pd.concat([full, partial], ignore_index=True).assign(
        label=lambda d: d["pct_change"].map(lambda v: f"{v:.1f}%"),
        sign=lambda d: np.where(d["pct_change"] >= 0, "increase", "decrease"),
    )


annual_industrial_chart_data = pd.concat(
    [
        annual_pct_chart_data(annual_ind_report, "Industrial Nighttime Lights"),
        annual_pct_chart_data(annual_noind_report, "Non-Industrial Nighttime Lights"),
    ],
    ignore_index=True,
)
write_ntl_sheet(annual_industrial_chart_data, "fig_2_2_ind_nonind_yoy")

annual_base = alt.Chart(annual_industrial_chart_data).encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelFontSize=9, titleFontSize=10)),
    y=alt.Y(
        "pct_change:Q",
        title="Luminosity % Change",
        axis=alt.Axis(labelFontSize=9, titleFontSize=10, grid=True, gridDash=[2, 2]),
    ),
    color=alt.Color(
        "series:N",
        title="",
        scale=alt.Scale(
            domain=["% change from PY", f"{REPORT_MAX_MONTH}M % change from PY"],
            range=["#754493", "#B7A4CB"],
        ),
        legend=alt.Legend(labelFontSize=9, symbolSize=80),
    ),
    tooltip=[
        alt.Tooltip("zone:N", title="Series"),
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("series:N", title="Measure"),
        alt.Tooltip("pct_change:Q", format="+.1f", title="% Change"),
    ],
)

bars = annual_base.mark_bar().encode(
    color=alt.condition(
        alt.datum.pct_change < 0,
        alt.value("#24768E"),
        alt.Color(
            "series:N",
            scale=alt.Scale(
                domain=["% change from PY", f"{REPORT_MAX_MONTH}M % change from PY"],
                range=["#754493", "#B7A4CB"],
            ),
            legend=alt.Legend(labelFontSize=9, symbolSize=80),
        ),
    )
)
labels = annual_base.mark_text(
    fontSize=8, color="#666666", dy=alt.expr("datum.pct_change >= 0 ? -5 : 12")
).encode(text="label:N")
zero = (
    alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#8A8A8A").encode(y="zero:Q")
)

ind_chart = (
    bars.transform_filter(alt.datum.zone == "Industrial Nighttime Lights")
    + labels.transform_filter(alt.datum.zone == "Industrial Nighttime Lights")
    + zero
).properties(
    width=420,
    height=260,
    title=alt.TitleParams(
        text="Industrial Nighttime Lights", fontSize=13, anchor="start"
    ),
)
noind_chart = (
    bars.transform_filter(alt.datum.zone == "Non-Industrial Nighttime Lights")
    + labels.transform_filter(alt.datum.zone == "Non-Industrial Nighttime Lights")
    + zero
).properties(
    width=420,
    height=260,
    title=alt.TitleParams(
        text="Non-Industrial Nighttime Lights", fontSize=13, anchor="start"
    ),
)

chart = (
    alt.hconcat(ind_chart, noind_chart)
    .resolve_scale(y="shared")
    .properties(
        title=alt.TitleParams(
            text="Annual % Change in Industrial and Non-Industrial Nighttime Lights",
            subtitle=f"Full-year change through {prev_year}; {REPORT_MAX_MONTH}-month change for {current_year}",
            fontSize=14,
            subtitleFontSize=10,
            anchor="start",
            offset=6,
        )
    )
)
chart = attaviz.add_caption(chart, "Source: NASA BlackMarble")
chart.save(NTL_REPORT_DIR / "fig-2-2-industrial-nonindustrial-yoy.png", scale_factor=3)
chart

alt.VConcatChart(...)

Overall, in 2025, there a 2% reduction in nightlights in industrial areas and 11% reduction in non industrial areas compared to 2024. 2026 has recovered in both industrial and non industrial zones

### 2.3 Monthly Trends in Industrial Areas (comparing 2024, 2025, 2026)

In [150]:
# Fig 2.3: Monthly Trends in Industrial and Non-Industrial Areas (comparing 2024, 2025, 2026)
PROJECT_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists() and (p / "notebooks").exists()
)
NTL_REPORT_DIR = PROJECT_ROOT / "docs" / "ntl_report"
NTL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
NTL_REPORT_XLSX = NTL_REPORT_DIR / "ntl-report-data.xlsx"

ind_monthly_data = ntl_ind_melted.assign(
    year_col=lambda d: d["year_col"].astype(str),
    month_name=lambda d: d["month"].map(lambda m: month_labels[int(m) - 1]),
).sort_values(["type", "year_col", "month"])

chart_specs = {
    "Industrial Zones (5km)": (
        "Industrial Zones Monthly Nighttime Lights",
        "fig-2-3-industrial-zones-monthly-nighttime-lights.png",
        "fig_2_3_industrial_monthly",
    ),
    "Non-Industrial Areas (5km)": (
        "Non-Industrial Areas Monthly Nighttime Lights",
        "fig-2-3-nonindustrial-areas-monthly-nighttime-lights.png",
        "fig_2_3_nonindustrial_monthly",
    ),
}


def monthly_ntl_chart(df, title):
    return (
        alt.Chart(df)
        .mark_line(point=False, strokeWidth=2)
        .encode(
            x=alt.X(
                "month:O",
                title="Month",
                sort=list(range(1, 13)),
                axis=alt.Axis(
                    labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value - 1]",
                    labelFontSize=8,
                    titleFontSize=9,
                ),
            ),
            y=alt.Y(
                "ntl_sum:Q",
                title="NTL Sum",
                axis=alt.Axis(
                    labelFontSize=8,
                    titleFontSize=9,
                    grid=True,
                    gridDash=[2, 2],
                    format="~s",
                ),
            ),
            color=alt.Color(
                "year_col:N",
                title="",
                scale=alt.Scale(range=["#025288", "#E3A763", "#754493"]),
                legend=alt.Legend(labelFontSize=8, symbolSize=60),
            ),
            tooltip=[
                alt.Tooltip("year_col:N", title="Year"),
                alt.Tooltip("month_name:N", title="Month"),
                alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
            ],
        )
        .properties(
            width=350,
            height=300,
            title=alt.TitleParams(
                text=title,
                subtitle="Monthly NTL sum, 2024-2026",
                fontSize=13,
                subtitleFontSize=10,
                anchor="start",
                offset=6,
            ),
        )
    )


monthly_charts = {}
for area_type, (title, filename, sheet_name) in chart_specs.items():
    chart_data = ind_monthly_data.query("type == @area_type").copy()
    write_ntl_sheet(chart_data, sheet_name)
    chart = attaviz.add_caption(
        monthly_ntl_chart(chart_data, title), "Source: NASA BlackMarble"
    )
    chart.save(NTL_REPORT_DIR / filename, scale_factor=3)
    monthly_charts[area_type] = chart

display(monthly_charts["Industrial Zones (5km)"])
display(monthly_charts["Non-Industrial Areas (5km)"])

alt.VConcatChart(...)

alt.VConcatChart(...)

Nightlights in industrial zones in the first four months have been higher across all months Jan-Apr in 2026 compared to 20265 and 2024. 

## 3. Nightlights in Regions

### 3.1 Quarterly Trends in Nightlights in Regions

In [151]:
# Fig 3.2: Regional % Change in Nightlights
REPORT_MAX_MONTH = max_month


def regional_ntl_change_data(df, year1, year2, value_col):
    grouped = (
        df.assign(
            date=lambda d: pd.to_datetime(d["date"]),
            year=lambda d: pd.to_datetime(d["date"]).dt.year,
            month=lambda d: pd.to_datetime(d["date"]).dt.month,
        )
        .query("month <= @REPORT_MAX_MONTH and year in [@year1, @year2]")
        .groupby(["ADM1_EN", "year"], as_index=False)
        .agg(ntl_value=(value_col, "sum"))
        .pivot(index="ADM1_EN", columns="year", values="ntl_value")
        .reset_index()
        .rename(columns={year1: "previous_value", year2: "current_value"})
        .assign(
            previous_year=year1,
            current_year=year2,
            pct_change=lambda d: np.where(
                d["previous_value"] > 0,
                (d["current_value"] - d["previous_value"]) / d["previous_value"] * 100,
                np.nan,
            ),
            comparison=f"{report_period_label} {year2} vs {year1}",
            label=lambda d: d["pct_change"].map(lambda v: f"{v:.1f}%"),
        )
        .sort_values("pct_change", ascending=False)
    )
    return grouped


regional_total_2026_2025 = regional_ntl_change_data(
    ntl_monthly_adm1, prev_year, current_year, "ntl_sum"
)
regional_total_2026_2024 = regional_ntl_change_data(
    ntl_monthly_adm1, prev_year - 1, current_year, "ntl_sum"
)
regional_total_chart_data = pd.concat(
    [regional_total_2026_2025, regional_total_2026_2024], ignore_index=True
)
base = alt.Chart(regional_total_chart_data).encode(
    x=alt.X(
        "pct_change:Q",
        title="% Change",
        axis=alt.Axis(labelFontSize=9, titleFontSize=10, grid=True, gridDash=[2, 2]),
    ),
    y=alt.Y(
        "ADM1_EN:N",
        sort="-x",
        title="Region (ADM1)",
        axis=alt.Axis(labelFontSize=8, titleFontSize=10),
    ),
    color=alt.condition(
        alt.datum.pct_change >= 0, alt.value("#754493"), alt.value("#24768E")
    ),
    tooltip=[
        alt.Tooltip("ADM1_EN:N", title="Region"),
        alt.Tooltip("comparison:N", title="Comparison"),
        alt.Tooltip("pct_change:Q", format="+.1f", title="% Change"),
    ],
)
bars = base.mark_bar()
labels = base.mark_text(
    fontSize=8,
    color="#666666",
    align=alt.expr("datum.pct_change >= 0 ? 'left' : 'right'"),
    dx=alt.expr("datum.pct_change >= 0 ? 3 : -3"),
).encode(text="label:N")
zero = (
    alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#8A8A8A").encode(x="zero:Q")
)
chart_2025 = (
    bars.transform_filter(
        alt.datum.comparison == f"{report_period_label} {current_year} vs {prev_year}"
    )
    + labels.transform_filter(
        alt.datum.comparison == f"{report_period_label} {current_year} vs {prev_year}"
    )
    + zero
).properties(
    width=360,
    height=380,
    title=alt.TitleParams(
        text=f"{report_period_label} {current_year} vs {prev_year}",
        fontSize=12,
        anchor="start",
    ),
)
chart_2024 = (
    bars.transform_filter(
        alt.datum.comparison
        == f"{report_period_label} {current_year} vs {prev_year - 1}"
    )
    + labels.transform_filter(
        alt.datum.comparison
        == f"{report_period_label} {current_year} vs {prev_year - 1}"
    )
    + zero
).properties(
    width=360,
    height=380,
    title=alt.TitleParams(
        text=f"{report_period_label} {current_year} vs {prev_year - 1}",
        fontSize=12,
        anchor="start",
    ),
)
chart = alt.hconcat(chart_2025, chart_2024).properties(
    title=alt.TitleParams(
        text="Regional % Change in Nightlights",
        subtitle=f"{report_period_subtitle} totals by ADM1",
        fontSize=14,
        subtitleFontSize=10,
        anchor="start",
        offset=6,
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)

Both Mandalay and Yangon saw ~20% increase in lights compared to 2024. 

In [152]:
# Fig 3.2: Regional % Change in Industrial Nightlights
REPORT_MAX_MONTH = max_month


def regional_ntl_change_data(df, year1, year2, value_col):
    grouped = (
        df.assign(
            date=lambda d: pd.to_datetime(d["date"]),
            year=lambda d: pd.to_datetime(d["date"]).dt.year,
            month=lambda d: pd.to_datetime(d["date"]).dt.month,
        )
        .query("month <= @REPORT_MAX_MONTH and year in [@year1, @year2]")
        .groupby(["ADM1_EN", "year"], as_index=False)
        .agg(ntl_value=(value_col, "sum"))
        .pivot(index="ADM1_EN", columns="year", values="ntl_value")
        .reset_index()
        .rename(columns={year1: "previous_value", year2: "current_value"})
        .assign(
            previous_year=year1,
            current_year=year2,
            pct_change=lambda d: np.where(
                d["previous_value"] > 0,
                (d["current_value"] - d["previous_value"]) / d["previous_value"] * 100,
                np.nan,
            ),
            comparison=f"{report_period_label} {year2} vs {year1}",
            label=lambda d: d["pct_change"].map(lambda v: f"{v:.1f}%"),
        )
        .sort_values("pct_change", ascending=False)
    )
    return grouped


regional_ind_2026_2025 = regional_ntl_change_data(
    ntl_monthly_adm1, prev_year, current_year, "ntl_ind_5km_sum"
)
regional_ind_2026_2024 = regional_ntl_change_data(
    ntl_monthly_adm1, prev_year - 1, current_year, "ntl_ind_5km_sum"
)
regional_industrial_chart_data = pd.concat(
    [regional_ind_2026_2025, regional_ind_2026_2024], ignore_index=True
)
write_ntl_sheet(regional_ind_2026_2025, "fig_3_2_ind_2026_vs_2025")
write_ntl_sheet(regional_ind_2026_2024, "fig_3_2_ind_2026_vs_2024")

regional_base = alt.Chart(regional_industrial_chart_data).encode(
    x=alt.X(
        "pct_change:Q",
        title="% Change",
        axis=alt.Axis(labelFontSize=9, titleFontSize=10, grid=True, gridDash=[2, 2]),
    ),
    y=alt.Y(
        "ADM1_EN:N",
        sort="-x",
        title="Region (ADM1)",
        axis=alt.Axis(labelFontSize=8, titleFontSize=10),
    ),
    color=alt.condition(
        alt.datum.pct_change >= 0, alt.value("#754493"), alt.value("#24768E")
    ),
    tooltip=[
        alt.Tooltip("ADM1_EN:N", title="Region"),
        alt.Tooltip("comparison:N", title="Comparison"),
        alt.Tooltip("previous_value:Q", format=",.0f", title="Previous NTL"),
        alt.Tooltip("current_value:Q", format=",.0f", title="Current NTL"),
        alt.Tooltip("pct_change:Q", format="+.1f", title="% Change"),
    ],
)
regional_bars = regional_base.mark_bar()
regional_labels = regional_base.mark_text(
    fontSize=8,
    color="#666666",
    align=alt.expr("datum.pct_change >= 0 ? 'left' : 'right'"),
    dx=alt.expr("datum.pct_change >= 0 ? 3 : -3"),
).encode(text="label:N")
regional_zero = (
    alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#8A8A8A").encode(x="zero:Q")
)

chart_2025 = (
    regional_bars.transform_filter(
        alt.datum.comparison == f"{report_period_label} {current_year} vs {prev_year}"
    )
    + regional_labels.transform_filter(
        alt.datum.comparison == f"{report_period_label} {current_year} vs {prev_year}"
    )
    + regional_zero
).properties(
    width=360,
    height=380,
    title=alt.TitleParams(
        text=f"{report_period_label} {current_year} vs {prev_year}",
        fontSize=12,
        anchor="start",
    ),
)
chart_2024 = (
    regional_bars.transform_filter(
        alt.datum.comparison
        == f"{report_period_label} {current_year} vs {prev_year - 1}"
    )
    + regional_labels.transform_filter(
        alt.datum.comparison
        == f"{report_period_label} {current_year} vs {prev_year - 1}"
    )
    + regional_zero
).properties(
    width=360,
    height=380,
    title=alt.TitleParams(
        text=f"{report_period_label} {current_year} vs {prev_year - 1}",
        fontSize=12,
        anchor="start",
    ),
)

chart = alt.hconcat(chart_2025, chart_2024).properties(
    title=alt.TitleParams(
        text="Regional % Change in Industrial Nightlights",
        subtitle=f"{report_period_subtitle} totals by ADM1",
        fontSize=14,
        subtitleFontSize=10,
        anchor="start",
        offset=6,
    )
)
chart = attaviz.add_caption(chart, "Source: NASA BlackMarble")
chart.save(
    NTL_REPORT_DIR / "fig-3-2-regional-industrial-nightlights.png", scale_factor=3
)
chart

alt.VConcatChart(...)

Most areas saw an increase in nighlights in industrial areas in 2026 compared to 2025 and 2024. However, Nay Pyi Taw saw a decrease in lights in 2026 compared to 2024. 

### 3.2 Monthly Trends in Regional Industrial Areas (comparing 2024, 2025, 2026)

In [156]:
# Fig 3.3b: Regional Monthly Nighttime Lights — Comparative Lines
regional_monthly_data = ntl_regional_comparison.assign(
    year_col=lambda d: d["year_col"].astype(str)
)
chart = (
    alt.Chart(regional_monthly_data)
    .mark_line(point=False, strokeWidth=1.6)
    .encode(
        x=alt.X(
            "month:O",
            title="",
            sort=list(range(1, 13)),
            axis=alt.Axis(
                labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value - 1]",
                labelFontSize=7,
            ),
        ),
        y=alt.Y(
            "ntl_sum:Q",
            title="NTL Sum",
            axis=alt.Axis(
                labelFontSize=7,
                titleFontSize=8,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        color=alt.Color(
            "year_col:N",
            title="",
            scale=alt.Scale(range=["#025288", "#E3A763", "#754493"]),
            legend=alt.Legend(labelFontSize=8, symbolSize=60),
        ),
        tooltip=[
            alt.Tooltip("ADM1_EN:N", title="Region"),
            alt.Tooltip("year_col:N", title="Year"),
            alt.Tooltip("month:O", title="Month"),
            alt.Tooltip("ntl_sum:Q", format=",.0f", title="NTL Sum"),
        ],
    )
    .properties(width=150, height=110)
    .facet(facet=alt.Facet("ADM1_EN:N", title="", sort=region_order_list), columns=4)
    .resolve_scale(y="independent")
    .properties(
        title=alt.TitleParams(
            text="Regional Monthly Nighttime Lights", fontSize=14, anchor="start"
        )
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)

In [155]:
# Fig 3.3b: Regional Industrial Monthly Nighttime Lights — Comparative Lines
regional_monthly_data = ntl_regional_comparison.assign(
    year_col=lambda d: d["year_col"].astype(str)
)
chart = (
    alt.Chart(regional_monthly_data)
    .mark_line(point=False, strokeWidth=1.6)
    .encode(
        x=alt.X(
            "month:O",
            title="",
            sort=list(range(1, 13)),
            axis=alt.Axis(
                labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value - 1]",
                labelFontSize=7,
            ),
        ),
        y=alt.Y(
            "ntl_ind_5km_sum:Q",
            title="NTL Sum",
            axis=alt.Axis(
                labelFontSize=7,
                titleFontSize=8,
                grid=True,
                gridDash=[2, 2],
                format="~s",
            ),
        ),
        color=alt.Color(
            "year_col:N",
            title="",
            scale=alt.Scale(range=["#025288", "#E3A763", "#754493"]),
            legend=alt.Legend(labelFontSize=8, symbolSize=60),
        ),
        tooltip=[
            alt.Tooltip("ADM1_EN:N", title="Region"),
            alt.Tooltip("year_col:N", title="Year"),
            alt.Tooltip("month:O", title="Month"),
            alt.Tooltip("ntl_ind_5km_sum:Q", format=",.0f", title="Industrial NTL Sum"),
        ],
    )
    .properties(width=150, height=110)
    .facet(facet=alt.Facet("ADM1_EN:N", title="", sort=region_order_list), columns=4)
    .resolve_scale(y="independent")
    .properties(
        title=alt.TitleParams(
            text=f"Industrial Zone Monthly Nighttime Lights ({prev_year - 1}-{current_year})",
            fontSize=14,
            anchor="start",
        )
    )
)
attaviz.add_caption(chart, "Source: NASA BlackMarble")

alt.VConcatChart(...)